[Crowell et al.](https://www.biorxiv.org/content/10.1101/2025.06.23.660674v1) data, available via [Zenodo](http://10.0.20.161/zenodo.15574384)

In [1]:
import dask
dask.config.set({"dataframe.query-planning": True})

In [2]:
import dask.dataframe as dd

In [3]:
import scanpy as sc
import gc
import scipy.sparse as sp

from tqdm.autonotebook import tqdm
from scipy.sparse import csr_matrix
from anndata.experimental import concat_on_disk

from cellina._spatial_utils import spatial_neighbors

/tmp/ipykernel_985667/3225973611.py:5: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
import sys

sys.path.append("../../scripts")

In [5]:
base_path = "/data2/a330d/datasets/crc"
joint_adata_path = f'{base_path}/processed/crc_cosmx_wt.h5ad'

## Create merged adata

In [6]:
data_list = ['110', '120', '210', '221', '231', '232', '242']

In [7]:
# Create tmp files with csr matrices (since the original files have csc, which causes issues with concatenation)
tmp_files = []

for idx in tqdm(data_list, desc="Converting to CSR"):
    ad = sc.read(f"{base_path}/raw_zenodo/crc_{idx}.h5ad")

    if sp.isspmatrix_csc(ad.X):
        ad.X = ad.X.tocsr()

    tmp = f"/data2/a330d/tmp/{idx}_csr.h5ad"
    ad.write(tmp)
    tmp_files.append(tmp)

Converting to CSR: 100%|██████████| 7/7 [01:56<00:00, 16.64s/it]


In [8]:
# Sequentially merge the rest (avoid exploding RAM)
for idx in tqdm(data_list, desc="Merging slides"):
    concat_on_disk(
    [f"/data2/a330d/tmp/{idx}_csr.h5ad" for idx in data_list],
    out_file=joint_adata_path,
    join="inner"
)

Merging slides: 100%|██████████| 7/7 [07:02<00:00, 60.35s/it]


## Read Object

In [7]:
adata = sc.read_h5ad(joint_adata_path)

/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


run - batch identifier (1 or 2)

sec - per-slide section identifier (1 or 2)

rep - pre-run donor identifier (1-2 for 1st, 1-4 for 2nd run)

did - slide identifier (run+rep, e.g., 12 for 1st run, 2nd slide)

sid - section identifier (run+rep+sec; xx0 for single-section slides)

pid - clinical sample identifier (unique except for sections 231/232)

fov - field of view (FOV) correspondence (n/a = all FOVs)

In [8]:
# Change back to csc - helps with downstream stuff
adata.X = adata.X.tocsc()

In [9]:
adata.obs[[
    'rep', # 2 replicates per donor?
    'sid', # sample id
    "did", # NOTE: donor id
    # To identify concordant microenvironments across sections and domains, 
    # we further clustered cells based on the subpopulation composition of their local neighborhood to obtain 10 spatial niches
    'ctx', 
 ]].drop_duplicates()

,rep,sid,did,ctx
c_1_1_1,1,110,11,N3
c_1_1_3,1,110,11,NaN
c_1_1_19,1,110,11,N6
c_1_1_134,1,110,11,N8
c_1_1_135,1,110,11,N1
...,...,...,...,...
c_4_35_443,4,242,24,N8
c_4_35_557,4,242,24,N6
c_4_36_10,4,242,24,N3
c_4_41_98,4,242,24,N1


In [10]:
adata.obsm['spatial'] = adata.obs[['CenterX_global_px', 'CenterY_global_px']].values

In [ ]:
slide_key = 'sid'

sc.pp.filter_genes(adata, min_cells=100)

/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [12]:
adata = adata[~adata.obs['ist'].isna()]
adata.obs['ist'] = adata.obs['ist'].astype(str)

/tmp/ipykernel_971048/3451605068.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['ist'] = adata.obs['ist'].astype(str)
/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
adata.layers['counts'] = adata.X.copy()

In [17]:
adata.write_h5ad(joint_adata_path)

## 2. Data Preprocessing

In [6]:
adata = sc.read_h5ad(joint_adata_path)

/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
# Compute spatial connectivity matrix
#from benchmark_pipeline import K_NN
K_NN = 50
label_col = "ist"
slide_key = "sid"

In [9]:
from _labels_to_coarse import LABEL_TO_COARSE

adata.obs["coarse_type"] = adata.obs[label_col].map(LABEL_TO_COARSE)
label_col = "coarse_type"

In [10]:
sc.pp.highly_variable_genes(adata, layer='counts', flavor='seurat_v3', n_top_genes=2000, subset=True)

/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Ligands

In [12]:
import pandas as pd
import numpy as np

import requests
import io

# read csv from link
# https://github.com/ventolab/cellphonedb-data/blob/master/data/interaction_input.csv
resource = requests.get('https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data/interaction_input.csv').content
resource = io.StringIO(resource.decode('utf-8'))
resource = pd.read_csv(resource, sep=',')
# keep only PPIs
resource = resource[resource['is_ppi']][['interactors']]
# replace + with _
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('+', '_'))
# if interactors contains two '-' replace the first one with '&
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('-', '&', 1) if x.count('-') == 2 else x)
# split by - and expand
resource = resource['interactors'].str.split('-', expand=True)
# replace & with - in the first column
resource[0] = resource[0].apply(lambda x: x.replace('&', '-'))
resource.columns = ['ligand', 'receptor']


In [13]:
# Split ligands on '_' to get list of subunits
resource['ligand'] = resource['ligand'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('ligand').reset_index(drop=True)
ligands = resource['ligand'].unique()

In [14]:
# Split receptors on '_' to get list of subunits
resource['receptor'] = resource['receptor'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('receptor').reset_index(drop=True)
receptors = resource['receptor'].unique()

In [15]:
ligands_in_data = [g for g in ligands if g in adata.var_names]
receptors_in_data = [g for g in receptors if g in adata.var_names]

In [17]:
len(ligands_in_data), len(receptors_in_data)

(161, 79)

In [18]:
use_ligands = False
use_receptors = False

neighbor_features = adata.var_names
if use_ligands:
    neighbor_features = ligands_in_data
if use_receptors:
    neighbor_features = neighbor_features + receptors_in_data

In [ ]:
# 1. Find which cell types are present in ALL slides
# Count how many unique slides there are
all_slides = adata.obs[slide_key].unique()
num_slides = len(all_slides)

# 2. Count slides per cell type
slides_per_celltype = adata.obs.groupby(label_col)[slide_key].nunique()

# 3. Select cell types that appear in all slides
celltypes_in_all_slides = slides_per_celltype[slides_per_celltype == num_slides].index.tolist()

# 4. Filter obs to keep only these cell types
adata = adata[adata.obs[label_col].isin(celltypes_in_all_slides)]

In [21]:
slides = adata.obs[slide_key].unique()
adata_out = None  # initialize output AnnData

for sid in tqdm(slides, desc="Adding spatial_x to slides"):
    ad_slide = adata[adata.obs[slide_key] == sid].copy()

    spatial_neighbors(ad_slide, bandwidth=np.inf, max_neighbours=K_NN, standardize=False)
    ad_slide.obs['neighbor'] = ad_slide.obsp['spatial_connectivities'][:,0].toarray().astype(np.float32)

    sc.pp.normalize_total(ad_slide, target_sum=1e4)
    sc.pp.log1p(ad_slide)
    ad_slide.obsm['spatial_x'] = ad_slide.obsp['spatial_connectivities'] @ ad_slide[:, neighbor_features].X / K_NN
    ad_slide.obsm['spatial_x'] = csr_matrix(ad_slide.obsm['spatial_x']).astype(np.float32)

    # Incremental concatenation
    if adata_out is None:
        adata_out = ad_slide
    else:
        adata_out = sc.concat([adata_out, ad_slide])

adata = adata_out

Adding spatial_x to slides:  29%|██▊       | 2/7 [02:37<06:49, 81.85s/it]/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Adding spatial_x to slides:  43%|████▎     | 3/7 [03:43<04:57, 74.39s/it]/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Adding spatial_x to slides:  57%|█████▋    | 4/7 [04:36<03:17, 65.98s/it]/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
/data/a330d/miniforge3/envs/cellina-base/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are

In [22]:
adata.X.max()

np.float64(9.210440366976517)

In [27]:
adata.obsm['spatial_x'].max()

np.float32(8.526325)

In [ ]:
#adata.layers['counts'] = adata.X.copy()
adata.X = adata.layers['counts'].copy()
adata.uns = ad_slide.uns.copy()

In [8]:
adata.obs["typ_clean"] = (
        adata.obs["typ"]
        .str.extract(r"(REF|TVA|CRC)", expand=False)
        )

In [9]:
adata.obs["typ_clean"] = adata.obs["typ_clean"].astype(str)
adata.obs["typ"] = adata.obs["typ"].astype(str)

In [10]:
test_samples = [221, 242]
batch_key = "sid"
sample_key = batch_key # NOTE: on this or by patient, not sure
celltype_key = 'coarse_type' # NOTE: I created this
niche_key = condition_key = 'typ_clean' # NOTE: ctx - this is some cluster on cell types TBD
n_holdout = 2
train_frac = 0.8

In [11]:
adata.uns['default_params'] = {
    # Data Specific Parameters
    "dataset_name": "cosmx_crc_wt",
    "K_NN": K_NN,
    "celltype_key": celltype_key,
    "niche_key": niche_key,
    "sample_key": sample_key,
    "batch_key": batch_key,
    # Split Parameters
    "n_holdout": n_holdout,
    "n_val_samples": False,
    "test_samples": test_samples,
    "train_frac": train_frac
}

In [12]:
adata.write_h5ad(joint_adata_path)